<a href="https://colab.research.google.com/github/yasminasouki/DataAnalysis/blob/dev/02_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
files.upload()   # choose kaggle.json
!mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

Mounted at /content/drive


Saving kaggle.json to kaggle.json


In [2]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/rsna
!unzip -q /content/rsna/rsna-pneumonia-detection-challenge.zip -d /content/rsna
!ls /content/rsna

100% 3.66G/3.66G [00:29<00:00, 132MB/s]

'GCP Credits Request Link - RSNA.txt'	  stage_2_test_images
 rsna-pneumonia-detection-challenge.zip   stage_2_train_images
 stage_2_detailed_class_info.csv	  stage_2_train_labels.csv
 stage_2_sample_submission.csv


In [3]:
!wget -q -O /content/rsna/mapping.json https://s3.amazonaws.com/east1.public.rsna.org/AI/2018/pneumonia-challenge-dataset-mappings_2018.json
!kaggle datasets download -d nih-chest-xrays/data -f Data_Entry_2017.csv -p /content/nih
!ls -la /content/rsna/mapping.json /content/nih

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/data
License(s): CC0-1.0
100% 924k/924k [00:00<00:00, 113MB/s]

-rw-r--r-- 1 root root 17587143 Jan 29  2019 /content/rsna/mapping.json

/content/nih:
total 936
drwxr-xr-x 2 root root   4096 Nov 10 19:13 .
drwxr-xr-x 1 root root   4096 Nov 10 19:13 ..
-rw-r--r-- 1 root root 946387 Sep 25  2019 Data_Entry_2017.csv.zip


In [4]:
import json, pandas as pd, glob

with open('/content/rsna/mapping.json') as f:
    mapping = json.load(f)
print(type(mapping), len(mapping))
print(mapping[:2] if isinstance(mapping, list) else list(mapping.items())[:2])

nih_file = glob.glob('/content/nih/Data_Entry_2017*')[0]
print("\nNIH file:", nih_file)
nih = pd.read_csv(nih_file)
print(nih.shape)
nih.head()

<class 'list'> 30000
[{'img_id': '00012773_000.png', 'subset_img_id': '1f6f5cb0-c537-4346-9300-98cf782d434f', 'subset_group': 0, 'subset_init_label': 2, 'orig_labels': ['Nodule'], 'StudyInstanceUID': '1.2.276.0.7230010.3.1.2.8323329.1475.1517874291.115797', 'SeriesInstanceUID': '1.2.276.0.7230010.3.1.3.8323329.1475.1517874291.115796', 'SOPInstanceUID': '1.2.276.0.7230010.3.1.4.8323329.1475.1517874291.115798', 'studyId': 'e40e6920-0eaecd2f-0a8be48e-eda11f63-f349078d', 'seriesId': '6196f40b-2345da85-b2a7370b-21e390ce-1c306a08', 'instanceId': 'a3f763d1-b9c99e7b-2b80f81f-67f78692-8787808b'}, {'img_id': '00030497_003.png', 'subset_img_id': '76c05ccc-eaf3-467b-abae-7f8f1a4aa6c9', 'subset_group': 0, 'subset_init_label': 1, 'orig_labels': ['Consolidation', 'Effusion', 'Infiltration'], 'StudyInstanceUID': '1.2.276.0.7230010.3.1.2.8323329.1476.1517874291.106014', 'SeriesInstanceUID': '1.2.276.0.7230010.3.1.3.8323329.1476.1517874291.106013', 'SOPInstanceUID': '1.2.276.0.7230010.3.1.4.8323329.1476

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [5]:
# RSNA ID -> NIH image name
map_df = pd.DataFrame(mapping)[["subset_img_id", "img_id"]]
map_df.columns = ["patientId", "nih_image"]

# NIH image name -> real patient ID
nih_small = nih[["Image Index", "Patient ID"]].rename(
    columns={"Image Index": "nih_image", "Patient ID": "real_patient_id"})

# Your metadata from last time (saved in Drive)
meta = pd.read_csv('/content/drive/MyDrive/pneumonia-project/metadata.csv')
meta = meta.merge(map_df, on="patientId", how="left").merge(nih_small, on="nih_image", how="left")

print("X-rays:", len(meta))
print("X-rays with no real patient found:", meta["real_patient_id"].isna().sum())
print("Real people:", meta["real_patient_id"].nunique())

counts = meta["real_patient_id"].value_counts()
print("People with more than one X-ray:", (counts > 1).sum())
print("Most X-rays for one person:", counts.max())
print("\nX-rays per person:\n", counts.describe().round(2))

X-rays: 26684
X-rays with no real patient found: 0
Real people: 11452
People with more than one X-ray: 4355
Most X-rays for one person: 65

X-rays per person:
 count    11452.00
mean         2.33
std          3.40
min          1.00
25%          1.00
50%          1.00
75%          2.00
max         65.00
Name: count, dtype: float64


In [6]:
classes_per_person = meta.groupby("real_patient_id")["class"].nunique()
print("People whose X-rays fall in different classes:", (classes_per_person > 1).sum())

meta.to_csv('/content/drive/MyDrive/pneumonia-project/metadata_with_patients.csv', index=False)
print("Saved")

People whose X-rays fall in different classes: 2697
Saved


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
meta = pd.read_csv('/content/drive/MyDrive/pneumonia-project/metadata_with_patients.csv')
print(meta.shape)
meta.head()

Mounted at /content/drive
(26684, 9)


,patientId,age,sex,view,height_px,width_px,class,nih_image,real_patient_id
0,b84112a1-ffcd-4897-baa1-652f4e460287,32,F,AP,1024,1024,Lung Opacity,00026546_004.png,26546
1,5b2e6149-2308-4f75-bf96-a0f062de13cc,22,M,PA,1024,1024,Normal,00011190_000.png,11190
2,a9b33024-ace7-424e-97c8-3082b8f5681f,41,M,PA,1024,1024,No Lung Opacity / Not Normal,00028876_017.png,28876
3,2139214a-4a50-4144-9c2d-eb4c929e1e7e,29,M,AP,1024,1024,Lung Opacity,00007018_040.png,7018
4,c236d6e2-1fca-40c5-a0fe-a91ff60883c9,24,F,AP,1024,1024,Lung Opacity,00006605_025.png,6605


In [5]:
# 1. At most 3 X-rays per person, chosen at random
capped = (meta.sample(frac=1, random_state=42)
              .groupby("real_patient_id").head(3)
              .reset_index(drop=True))
print("After cap:", len(capped), "X-rays from", capped["real_patient_id"].nunique(), "people")
print("Classes after cap (%):")
print((capped["class"].value_counts(normalize=True) * 100).round(1))
print("AP view after cap (%):", round((capped["view"] == "AP").mean() * 100, 1))

# 2. Keep all pneumonia X-rays after the cap, fill the rest equally
lung = capped[capped["class"] == "Lung Opacity"]
others_n = (9000 - len(lung)) // 2

normal = capped[capped["class"] == "Normal"].sample(n=others_n, random_state=42)
not_normal = capped[capped["class"] == "No Lung Opacity / Not Normal"].sample(n=others_n, random_state=42)

subset = pd.concat([lung, normal, not_normal]).reset_index(drop=True)
print("\nSubset:", len(subset), "X-rays from", subset["real_patient_id"].nunique(), "people")
print("Classes in subset (%):")
print((subset["class"].value_counts(normalize=True) * 100).round(1))
print("AP view in subset (%):", round((subset["view"] == "AP").mean() * 100, 1))

After cap: 18351 X-rays from 11452 people
Classes after cap (%):
class
Normal                          42.9
No Lung Opacity / Not Normal    41.9
Lung Opacity                    15.2
Name: proportion, dtype: float64
AP view after cap (%): 34.7

Subset: 8999 X-rays from 6682 people
Classes in subset (%):
class
Normal                          34.5
No Lung Opacity / Not Normal    34.5
Lung Opacity                    30.9
Name: proportion, dtype: float64
AP view in subset (%): 40.5


In [6]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# Test set = about 20% of people
trainval_idx, test_idx = next(sgkf.split(subset, subset["class"], groups=subset["real_patient_id"]))
trainval, test = subset.iloc[trainval_idx], subset.iloc[test_idx]

# Validation = about 20% of the remaining people
train_idx, val_idx = next(sgkf.split(trainval, trainval["class"], groups=trainval["real_patient_id"]))
train, val = trainval.iloc[train_idx], trainval.iloc[val_idx]

for name, d in [("Train", train), ("Validation", val), ("Test", test)]:
    print(f"\n{name}: {len(d)} X-rays, {d['real_patient_id'].nunique()} people")
    print("  Classes (%):", (d["class"].value_counts(normalize=True) * 100).round(1).to_dict())
    print("  AP view (%):", round((d["view"] == "AP").mean() * 100, 1))

print("\nPeople in both train and test:", len(set(train.real_patient_id) & set(test.real_patient_id)))
print("People in both train and validation:", len(set(train.real_patient_id) & set(val.real_patient_id)))
print("People in both validation and test:", len(set(val.real_patient_id) & set(test.real_patient_id)))

split = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
split.to_csv('/content/drive/MyDrive/pneumonia-project/split.csv', index=False)
print("\nSaved split.csv:", split.shape)


Train: 5688 X-rays, 4233 people
  Classes (%): {'No Lung Opacity / Not Normal': 34.9, 'Normal': 34.1, 'Lung Opacity': 31.0}
  AP view (%): 40.9

Validation: 1442 X-rays, 1075 people
  Classes (%): {'Normal': 35.3, 'No Lung Opacity / Not Normal': 34.9, 'Lung Opacity': 29.8}
  AP view (%): 39.1

Test: 1869 X-rays, 1374 people
  Classes (%): {'Normal': 35.1, 'No Lung Opacity / Not Normal': 33.2, 'Lung Opacity': 31.7}
  AP view (%): 40.3

People in both train and test: 0
People in both train and validation: 0
People in both validation and test: 0

Saved split.csv: (8999, 10)


In [8]:
!pip install -q pydicom
import os, numpy as np, pydicom
from PIL import Image
from tqdm import tqdm

# Re-download the images if Colab wiped them
if not os.path.exists('/content/rsna/stage_2_train_images'):
    !mkdir -p ~/.kaggle
    print("Images missing: upload kaggle.json again")
    from google.colab import files
    files.upload()
    !mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c rsna-pneumonia-detection-challenge -p /content/rsna
    !unzip -q /content/rsna/rsna-pneumonia-detection-challenge.zip -d /content/rsna

img_dir = '/content/rsna/stage_2_train_images'
images = np.zeros((len(split), 224, 224), dtype=np.uint8)

for i, pid in enumerate(tqdm(split["patientId"])):
    img = pydicom.dcmread(f"{img_dir}/{pid}.dcm").pixel_array
    images[i] = np.array(Image.fromarray(img).resize((224, 224), Image.BILINEAR))

np.savez_compressed('/content/drive/MyDrive/pneumonia-project/images_224.npz',
                    images=images, patientId=split["patientId"].values)
print("Saved:", images.shape)

Images missing: upload kaggle.json again


Saving kaggle.json to kaggle.json
100% 3.66G/3.66G [00:45<00:00, 87.2MB/s]



100%|██████████| 8999/8999 [01:10<00:00, 127.65it/s]


Saved: (8999, 224, 224)


In [9]:
rows = []
for name in ["train", "val", "test"]:
    d = split[split["split"] == name]
    pct = (d["class"].value_counts(normalize=True) * 100).round(1)
    rows.append({
        "Set": name,
        "X-rays": len(d),
        "People": d["real_patient_id"].nunique(),
        "Normal (%)": pct.get("Normal"),
        "Not Normal (%)": pct.get("No Lung Opacity / Not Normal"),
        "Lung Opacity (%)": pct.get("Lung Opacity"),
        "AP view (%)": round((d["view"] == "AP").mean() * 100, 1),
    })
pd.DataFrame(rows)

,Set,X-rays,People,Normal (%),Not Normal (%),Lung Opacity (%),AP view (%)
0,train,5688,4233,34.1,34.9,31.0,40.9
1,val,1442,1075,35.3,34.9,29.8,39.1
2,test,1869,1374,35.1,33.2,31.7,40.3
